In [1]:
import json
import pathlib
import os
import numpy as np
import pandas as pd


# Robustly find a file among common Kaggle directories
def find_path(*candidates):
    # direct checks
    for p in candidates:
        if p.is_file():
            return p
    # search in typical Kaggle input locations
    search_dirs = [
        pathlib.Path("."),
        pathlib.Path(".."),
        pathlib.Path("/kaggle/input"),
        pathlib.Path("/kaggle/working"),
    ]
    for base in search_dirs:
        for p in candidates:
            possible = base / p
            if possible.is_file():
                return possible
            # also look inside subfolders like stanford-covid-vaccine
            possible = base / "stanford-covid-vaccine" / p.name
            if possible.is_file():
                return possible
    raise FileNotFoundError(f"None of the candidate files exist: {candidates}")


data_dir = pathlib.Path("data")

train_path = find_path(
    data_dir / "train.json",
    pathlib.Path("train.json"),
    pathlib.Path("stanford-covid-vaccine") / "train.json",
)
test_path = find_path(
    data_dir / "test.json",
    pathlib.Path("test.json"),
    pathlib.Path("stanford-covid-vaccine") / "test.json",
)


def load_json(path):
    """Load a JSON file that may be a single array or line‑delimited objects."""
    with open(path, "r") as f:
        txt = f.read().strip()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        objs = []
        for line in txt.splitlines():
            line = line.strip()
            if line:
                objs.append(json.loads(line))
        return objs


train_data = load_json(train_path)
test_data = load_json(test_path)

targets = ["reactivity", "deg_Mg_pH10", "deg_pH10", "deg_Mg_50C", "deg_50C"]
base2idx = {"A": 0, "C": 1, "G": 2, "U": 3}
num_scored = max(entry["seq_scored"] for entry in train_data)  # should be 68
num_bases = len(base2idx)



In [2]:
# Compute per‑position overall means and per‑base means from training data
means_train = {t: np.zeros(num_scored, dtype=float) for t in targets}
counts_pos = np.zeros(num_scored, dtype=int)

base_means = {t: np.zeros((num_scored, num_bases), dtype=float) for t in targets}
base_counts = np.zeros((num_scored, num_bases), dtype=int)

for entry in train_data:
    seq = entry["sequence"]
    seq_scored = entry["seq_scored"]
    for pos in range(seq_scored):
        counts_pos[pos] += 1
        base = seq[pos]
        if base not in base2idx:
            continue
        b_idx = base2idx[base]
        base_counts[pos, b_idx] += 1
        for t in targets:
            val = float(entry[t][pos])
            means_train[t][pos] += val
            base_means[t][pos, b_idx] += val

# Finalise overall and base‑wise means
for t in targets:
    means_train[t] = np.where(counts_pos > 0, means_train[t] / counts_pos, 0.0)
    for b_idx in range(num_bases):
        cnt = base_counts[:, b_idx]
        mask = cnt > 0
        base_means[t][mask, b_idx] = base_means[t][mask, b_idx] / cnt[mask]

# Smoothed version: fall back to overall mean when a base has no count
smoothed_base_means_train = {}
for t in targets:
    arr = np.where(base_counts > 0, base_means[t], means_train[t][:, None])
    smoothed_base_means_train[t] = arr



In [3]:
# Simple linear calibration (scale + offset) using an internal validation split
rng = np.random.RandomState(42)
indices = np.arange(len(train_data))
rng.shuffle(indices)
split = int(0.8 * len(train_data))
val_idx = indices[split:]
val_split = [train_data[i] for i in val_idx]

calibration = {}  # t -> (a_array, b_array)
for t in targets:
    a_arr = np.ones(num_scored, dtype=float)
    b_arr = np.zeros(num_scored, dtype=float)
    for pos in range(num_scored):
        true_vals = []
        pred_vals = []
        for entry in val_split:
            if pos >= entry["seq_scored"]:
                continue
            true_arr = np.asarray(entry[t], dtype=float)
            seq = entry["sequence"]
            base = seq[pos]
            if base in base2idx and base_counts[pos, base2idx[base]] > 0:
                pred = smoothed_base_means_train[t][pos, base2idx[base]]
            else:
                pred = means_train[t][pos]
            true_vals.append(true_arr[pos])
            pred_vals.append(pred)
        true_vals = np.array(true_vals, dtype=float)
        pred_vals = np.array(pred_vals, dtype=float)

        if len(true_vals) < 2 or np.sum(pred_vals**2) == 0:
            a, b = 1.0, 0.0
        else:
            a = np.sum(true_vals * pred_vals) / np.sum(pred_vals**2)
            b = true_vals.mean() - a * pred_vals.mean()
            if a <= 0:
                a, b = 1.0, 0.0
        a_arr[pos] = a
        b_arr[pos] = b
    calibration[t] = (a_arr, b_arr)



In [4]:
# Generate predictions for every position in the test set and write submission
rows = []
for entry in test_data:
    seq = entry["sequence"]
    seq_scored = entry["seq_scored"]
    seq_len = entry["seq_length"]  # should be 107
    for pos in range(seq_len):
        id_seqpos = f"{entry['id']}_{pos}"
        row = {"id_seqpos": id_seqpos}
        for t in targets:
            if pos < seq_scored:
                base = seq[pos] if pos < len(seq) else None
                if base in base2idx and base_counts[pos, base2idx[base]] > 0:
                    pred = smoothed_base_means_train[t][pos, base2idx[base]]
                else:
                    pred = means_train[t][pos]
            else:
                pred = means_train[t].mean()
            row[t] = pred
        rows.append(row)

pred_df = pd.DataFrame(rows)

# Apply calibration per position
pred_df["pos"] = pred_df["id_seqpos"].apply(lambda x: int(x.split("_")[-1]))
pos_idx = np.minimum(pred_df["pos"].values, num_scored - 1)

for t in targets:
    a_arr, b_arr = calibration.get(t, (np.ones(num_scored), np.zeros(num_scored)))
    pred_df[t] = pred_df[t].values * a_arr[pos_idx] + b_arr[pos_idx]

pred_df.drop(columns=["pos"], inplace=True)

submission = pred_df[["id_seqpos"] + targets]

out_path = pathlib.Path("submission.csv")
submission.to_csv(out_path, index=False)
print(f"Submission written to {out_path.resolve()}, shape={submission.shape}")

Submission written to /kaggle/working/submission.csv, shape=(25680, 6)
